# Module 5.5 — Advanced Asyncio

### Python Mastery · Track 5: Concurrency & Asynchronous Programming

With the foundations in place, this module covers the tools that make asyncio practical for real work: asynchronous context managers and iterators, synchronisation primitives that work with coroutines, structured concurrency through `TaskGroup`, and the producer-consumer pattern with an async queue.

**Running async code in a notebook.** As in Module 5.4, the executable cells use top-level `await` rather than `asyncio.run`, because a notebook already has a running event loop.

### Learning objectives

After completing this module you will be able to:

1. Write and use asynchronous context managers (`async with`).
2. Write and use asynchronous iterators and generators (`async for`).
3. Coordinate coroutines with `asyncio.Lock`, `Semaphore`, and `Queue`.
4. Use `asyncio.TaskGroup` for structured concurrency (Python 3.11+).
5. Limit concurrency and aggregate results in async workflows.

**Prerequisites:** Tracks 1 to 4, and Modules 5.1 to 5.4.

---

## Part 1 · Asynchronous Context Managers (`async with`)

A regular context manager (Module 3.3) cannot pause to await setup or cleanup. An **async context manager** can: it implements `__aenter__` and `__aexit__`, both coroutines, and is used with `async with`. This suits resources whose acquisition or release involves input/output, such as opening a connection.

In [ ]:
import asyncio

class Connection:
    """An async context manager simulating a connection that opens and closes with awaits."""
    async def __aenter__(self):
        await asyncio.sleep(0.01)        # setup may involve waiting
        self.events = ["connected"]
        return self

    async def __aexit__(self, exc_type, exc_value, tb):
        await asyncio.sleep(0.01)        # cleanup may involve waiting too
        self.events.append("closed")
        return False

    async def query(self, q):
        await asyncio.sleep(0.01)
        self.events.append(f"queried {q}")
        return f"result of {q}"

async with Connection() as conn:
    print(await conn.query("SELECT 1"))

print("lifecycle:", conn.events)         # connected -> queried -> closed

## Part 2 · Asynchronous Iterators (`async for`)

An **async iterator** produces values that each require awaiting, ideal for streaming data that arrives over time, such as lines from a network source. Define one with `__aiter__` and an `__anext__` coroutine, or more simply with an **async generator**: an `async def` function that uses `yield`. Consume either with `async for`.

In [ ]:
import asyncio

async def stream_numbers(n):
    """An async generator yielding values with a pause between each."""
    for i in range(1, n + 1):
        await asyncio.sleep(0.01)        # each value 'arrives' after a delay
        yield i

# Consume the asynchronous stream with async for.
collected = []
async for value in stream_numbers(5):
    collected.append(value)
print("streamed:", collected)

# Async comprehensions work too.
squares = [x * x async for x in stream_numbers(4)]
print("squares via async comprehension:", squares)

## Part 3 · Async Synchronisation: `Lock`, `Semaphore`, `Queue`

Even though asyncio runs one coroutine at a time, coordination is still needed: to protect a multi-step update across awaits, or to limit how many coroutines do something at once. asyncio provides async versions of the familiar primitives. They are awaited rather than blocking the thread.

An `asyncio.Semaphore` is especially common for limiting concurrency, for example capping simultaneous network requests.

In [ ]:
import asyncio

# Limit to 2 concurrent "downloads" using an async semaphore.
semaphore = asyncio.Semaphore(2)
active = 0
peak = 0

async def download(item):
    global active, peak
    async with semaphore:                # at most 2 coroutines inside at once
        active += 1
        peak = max(peak, active)
        await asyncio.sleep(0.02)
        active -= 1
        return item

results = await asyncio.gather(*(download(f"file{i}") for i in range(6)))
print("downloaded:", results)
print("peak concurrent downloads:", peak, "(capped at 2)")

An `asyncio.Queue` enables the producer-consumer pattern within async code. Producers `await queue.put(...)` and consumers `await queue.get()`, all without blocking the event loop.

In [ ]:
import asyncio

async def producer(queue, n):
    for i in range(1, n + 1):
        await queue.put(i)
    await queue.put(None)                # sentinel to stop the consumer

async def consumer(queue, results):
    while True:
        item = await queue.get()
        if item is None:
            break
        results.append(item * item)

queue = asyncio.Queue()
results = []
await asyncio.gather(producer(queue, 5), consumer(queue, results))
print("processed:", results)

## Part 4 · Structured Concurrency with `TaskGroup`

Managing many tasks by hand can leak tasks or hide errors. `asyncio.TaskGroup` (Python 3.11+) provides **structured concurrency**: tasks are created inside an `async with` block, the block does not exit until all of them finish, and if any task fails the rest are cancelled and the errors are raised together. This is now the recommended way to run a group of related tasks.

In [ ]:
import asyncio

async def fetch(item, delay):
    await asyncio.sleep(delay)
    return f"data:{item}"

results = []
async with asyncio.TaskGroup() as group:     # the block waits for all tasks
    for i, delay in enumerate([0.02, 0.01, 0.03]):
        group.create_task(_collect(item=i, delay=delay, store=results)) \
            if False else None                 # placeholder to keep structure clear

# Cleaner form: create tasks and read their results after the group completes.
tasks = []
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(fetch(f"item{i}", d))
             for i, d in enumerate([0.02, 0.01, 0.03])]
# By here, every task has finished successfully.
print("all results:", [t.result() for t in tasks])

When a task in a group fails, the group cancels the others and raises the errors as an `ExceptionGroup`, which you handle with `except*` (introduced in Module 1.5). This makes failures in concurrent code visible and contained rather than silently lost.

In [ ]:
import asyncio

async def ok(n):
    await asyncio.sleep(0.01)
    return n

async def fails():
    await asyncio.sleep(0.01)
    raise ValueError("task failed")

try:
    async with asyncio.TaskGroup() as group:
        group.create_task(ok(1))
        group.create_task(fails())
        group.create_task(ok(2))
except* ValueError as eg:
    print("caught grouped failure:", [str(e) for e in eg.exceptions])

## Part 5 · A Realistic Pattern: Bounded Concurrent Fetching

Putting the pieces together: a common real task is fetching many resources concurrently but with a cap on simultaneous requests, then aggregating the results. This combines a semaphore (to bound concurrency) with gather (to run and collect), and is the backbone of well-behaved async input/output code.

In [ ]:
import asyncio

async def fetch_one(item, semaphore):
    async with semaphore:                # respect the concurrency limit
        await asyncio.sleep(0.02)        # simulate the request
        return item, len(item)           # pretend: the size of the response

async def fetch_many(items, limit):
    semaphore = asyncio.Semaphore(limit)
    tasks = [fetch_one(item, semaphore) for item in items]
    return await asyncio.gather(*tasks)

items = ["alpha", "beta", "gamma", "delta", "epsilon"]
results = await fetch_many(items, limit=2)
print("results:", results)
print("total size:", sum(size for _, size in results))

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — An async generator (Easy)

In [ ]:
import asyncio

async def ticks(n):
    for i in range(n):
        await asyncio.sleep(0.01)
        yield i

values = [v async for v in ticks(4)]
print(values)

### Example 2 — An async context manager (Easy)

In [ ]:
import asyncio

class Session:
    async def __aenter__(self):
        self.log = ["open"]
        return self
    async def __aexit__(self, *exc):
        self.log.append("close")
        return False

async with Session() as s:
    s.log.append("work")
print(s.log)

### Example 3 — Limiting concurrency with a semaphore (Medium)

In [ ]:
import asyncio

sem = asyncio.Semaphore(3)
inside = 0
peak = 0

async def task(n):
    global inside, peak
    async with sem:
        inside += 1
        peak = max(peak, inside)
        await asyncio.sleep(0.02)
        inside -= 1
    return n

await asyncio.gather(*(task(n) for n in range(9)))
print("peak concurrent:", peak)

### Example 4 — Producer-consumer with an async queue (Medium)

In [ ]:
import asyncio

async def produce(q):
    for word in ["a", "bb", "ccc"]:
        await q.put(word)
    await q.put(None)

async def consume(q, out):
    while True:
        word = await q.get()
        if word is None:
            break
        out.append(len(word))

q = asyncio.Queue()
lengths = []
await asyncio.gather(produce(q), consume(q, lengths))
print("lengths:", lengths)

### Example 5 — Structured concurrency with TaskGroup (Difficult)

In [ ]:
import asyncio

async def fetch(n):
    await asyncio.sleep(0.02 - n * 0.005)
    return f"result-{n}"

async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(fetch(n)) for n in range(4)]

print("collected:", [t.result() for t in tasks])

### Example 6 — Handling a failing task in a group (Difficult)

In [ ]:
import asyncio

async def step(n):
    await asyncio.sleep(0.01)
    if n == 2:
        raise RuntimeError(f"step {n} broke")
    return n

caught = None
try:
    async with asyncio.TaskGroup() as group:
        for n in range(4):
            group.create_task(step(n))
except* RuntimeError as eg:
    caught = [str(e) for e in eg.exceptions]

print("errors surfaced:", caught)

---

## Exercises

Use top-level `await` (not `asyncio.run`) in your answers.

**Exercise 1 (Easy).** Write an async generator `countdown(n)` that yields `n, n-1, ..., 1` with a short pause between each, and collect its values with an `async for` into a list.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Write an async context manager `Timer` whose `__aenter__` records `"start"` and `__aexit__` records `"stop"` in a list. Use it and print the list.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Use an `asyncio.Semaphore` with a limit of 2 to run six short tasks, tracking and printing the peak number that ran at the same time.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Use an `asyncio.Queue` with a producer that puts the numbers 1 to 4 plus a sentinel, and a consumer that collects their doubles. Run both with `gather` and print the collected list.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Use `asyncio.TaskGroup` to run four coroutines that each return their input squared, then print the list of results read from the tasks after the group completes.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import asyncio

async def countdown(n):
    while n > 0:
        await asyncio.sleep(0.01)
        yield n
        n -= 1

values = [v async for v in countdown(5)]
print(values)

**Solution 2**

In [ ]:
import asyncio

class Timer:
    async def __aenter__(self):
        self.log = ["start"]
        return self
    async def __aexit__(self, *exc):
        self.log.append("stop")
        return False

async with Timer() as t:
    pass
print(t.log)

**Solution 3**

In [ ]:
import asyncio

sem = asyncio.Semaphore(2)
inside = 0
peak = 0

async def task():
    global inside, peak
    async with sem:
        inside += 1
        peak = max(peak, inside)
        await asyncio.sleep(0.02)
        inside -= 1

await asyncio.gather(*(task() for _ in range(6)))
print("peak:", peak)

**Solution 4**

In [ ]:
import asyncio

async def producer(q):
    for n in range(1, 5):
        await q.put(n)
    await q.put(None)

async def consumer(q, out):
    while True:
        n = await q.get()
        if n is None:
            break
        out.append(n * 2)

q = asyncio.Queue()
out = []
await asyncio.gather(producer(q), consumer(q, out))
print(out)

**Solution 5**

In [ ]:
import asyncio

async def square(n):
    await asyncio.sleep(0.01)
    return n * n

async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(square(n)) for n in range(4)]

print([t.result() for t in tasks])

---

## Summary and Key Points

- Async context managers implement `__aenter__` and `__aexit__` (coroutines) and are used with `async with`, suiting resources whose setup or cleanup involves awaiting.
- Async iterators and async generators (`async def` with `yield`) produce values that require awaiting, consumed with `async for` or async comprehensions, ideal for streaming data.
- asyncio provides awaitable synchronisation: `Lock` protects multi-step updates across awaits, `Semaphore` bounds concurrency, and `Queue` enables async producer-consumer flows.
- `asyncio.TaskGroup` (Python 3.11+) gives structured concurrency: tasks created in an `async with` block all complete before exit, and a failure cancels the rest and raises an `ExceptionGroup` handled with `except*`.
- A bounded-concurrency fetch (semaphore plus gather) is the standard pattern for well-behaved async input/output, running many requests while capping how many happen at once.

### Next module: 5.6 — Distributed Task Concepts

The next module covers the ideas behind scaling work beyond one machine: task queues, idempotency, retries with backoff, dead-letter queues, and designing for eventual consistency.